# Full manual lineage: Power plant emissions

Run each stage and inspect its output before continuing. The pinned 2026-09-26 publisher response and verified rebuild are supplied. Set `USE_PINNED_RAW = False` only for a deliberate fresh-source run; its values may differ. All processing and formulas are visible in this notebook. No portfolio module, launcher, or configuration file is required.


## 1 Source

Inspect the publisher URL and query below: https://www.epa.gov/system/files/documents/2025-06/egrid2023_data_rev2.xlsx. The included raw response is a dated publisher payload with a SHA-256 digest, not the unsupported older portfolio snapshot. To acquire a new response, set `USE_PINNED_RAW = False` before running.

**Check before continuing:** publisher, selection, project folder, and pinned-response location must match this project; any new response must be preserved with its retrieval metadata.


In [1]:
import hashlib
import json
from datetime import UTC, date, datetime
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile
import numpy as np
import pandas as pd
import requests
from IPython.display import display

SLUG = "08_power_plant_emissions"
FOLDER = SLUG.split("_", 1)[1]
SOURCE_URL = "https://www.epa.gov/system/files/documents/2025-06/egrid2023_data_rev2.xlsx"
SNAPSHOT_DATE_OVERRIDE = None  # Leave None for the dated publisher response.
USE_PINNED_RAW = True
ROOT = next((candidate for candidate in (Path.cwd(), *Path.cwd().parents)
             if (candidate / "projects" / FOLDER / "README.md").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the extracted portfolio.")
PROJECT = ROOT / "projects" / FOLDER
RAW_DIR = PROJECT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
FETCH_LOG = []
PINNED_RESPONSES = {'6b3d9b8d303ff032': '895cd81dd8662406189ad8adf5a2578dcb362cdb5cff7cca81b10ee6bd2447c6'}
print("Project:", PROJECT.relative_to(ROOT).as_posix())
print("Publisher:", SOURCE_URL)
print("Pinned response folder:", RAW_DIR.relative_to(ROOT).as_posix())

"""Fuel/technology comparisons of historical output-normalized CO2."""


import numpy as np
import pandas as pd


def generator_technology(generators: pd.DataFrame) -> pd.DataFrame:
    """Classify dominant technology from positive reported generator output.

    Negative and missing generation do not enter the mix denominator. The
    result is descriptive of reported positive output, not installed capacity.
    """
    x = generators.copy()
    if x[["ORISPL", "GENID"]].isna().any().any() or x.duplicated(["ORISPL", "GENID"]).any():
        raise ValueError("Generator identifiers must be observed and unique")
    x["GENNTAN"] = pd.to_numeric(x.GENNTAN, errors="coerce")
    x["PRMVR"] = x.PRMVR.fillna("Unknown")
    x = x.loc[x.GENNTAN.gt(0)]
    mix = x.groupby(["ORISPL", "PRMVR"], as_index=False).GENNTAN.sum()
    mix["positive_output_share"] = mix.GENNTAN / mix.groupby("ORISPL").GENNTAN.transform("sum")
    top = mix.sort_values(
        ["ORISPL", "positive_output_share", "PRMVR"], ascending=[True, False, True]
    ).drop_duplicates("ORISPL")
    top["technology"] = top.PRMVR.where(top.positive_output_share.ge(0.8), "Mixed")
    return top[["ORISPL", "technology", "positive_output_share"]]


def peer_scenarios(plants: pd.DataFrame, *, minimum_peers: int = 5) -> pd.DataFrame:
    """Quantile scenarios, retaining output and never claiming feasibility.

    Plant-count quantiles describe a typical peer plant, not a fleet-weighted
    benchmark. Unknown technology is excluded from benchmark estimation.
    Intensity and scenario output use short tons/MWh and short tons respectively;
    the separate presentation field ``co2_lb_mwh`` is converted with 2,000
    pounds per short ton in the transform.
    """
    if minimum_peers < 2:
        raise ValueError("At least two peers are required")
    x = plants.copy()
    if x.plant_id.isna().any() or x.plant_id.duplicated().any():
        raise ValueError("Plant grain must be unique")
    for field in ["net_generation_mwh", "co2_tons"]:
        x[field] = pd.to_numeric(x[field], errors="coerce")
    valid = x.net_generation_mwh.gt(0) & x.co2_tons.ge(0)
    x["co2_tons_per_mwh"] = (x.co2_tons / x.net_generation_mwh).where(valid)
    known = x.technology.notna() & x.technology.ne("Unknown") & x.fuel.notna()
    rows = []
    for quantile in [0.25, 0.5, 0.75]:
        peers = (
            x.loc[known]
            .groupby(["fuel", "technology"])
            .co2_tons_per_mwh.agg(
                peer_count="count", benchmark=lambda values, q=quantile: values.quantile(q)
            )
            .reset_index()
        )
        y = x.merge(peers, on=["fuel", "technology"], how="left", validate="many_to_one")
        y["benchmark"] = y.benchmark.where(y.peer_count.ge(minimum_peers))
        y["scenario_co2_tons"] = (
            (y.co2_tons_per_mwh - y.benchmark).clip(lower=0) * y.net_generation_mwh
        ).where(valid)
        y["benchmark_quantile"] = quantile
        y = y.sort_values(["scenario_co2_tons", "plant_id"], ascending=[False, True])
        total = y.scenario_co2_tons.sum(min_count=1)
        y["cumulative_opportunity_share"] = y.scenario_co2_tons.cumsum() / (
            total if total > 0 else np.nan
        )
        rows.append(y)
    return pd.concat(rows, ignore_index=True)

def preserve_response(
    response: requests.Response,
    raw_root: Path,
    *,
    filename: str,
    publisher: str,
    **metadata: object,
) -> Path:
    response.raise_for_status()
    if Path(filename).name != filename:
        raise ValueError("Snapshot filename must be a basename")
    payload = response.content
    if not payload:
        raise ValueError("Cannot preserve an empty source snapshot")
    digest = hashlib.sha256(payload).hexdigest()
    directory = raw_root / digest
    directory.mkdir(parents=True, exist_ok=True)
    destination = directory / filename
    if destination.exists() and hashlib.sha256(destination.read_bytes()).hexdigest() != digest:
        raise ValueError("Content-addressed snapshot collision")
    if not destination.exists():
        destination.write_bytes(payload)
    record = {
        **metadata,
        "publisher": publisher,
        "direct_url": response.url,
        "sha256": digest,
        "bytes": len(payload),
        "accessed_at": datetime.now(UTC).isoformat(),
    }
    # Response headers are deliberately excluded from public provenance.
    metadata_path = directory / "metadata.json"
    if metadata_path.exists():
        saved = json.loads(metadata_path.read_text(encoding="utf-8"))
        if saved.get("sha256") != digest or saved.get("bytes") != len(payload):
            raise ValueError("Content-addressed snapshot metadata mismatch")
    else:
        metadata_path.write_text(json.dumps(record, indent=2), encoding="utf-8")
    return destination

def project_dir(slug):
    return ROOT / "projects" / slug.split("_", 1)[1]

def _get(url, **kwargs):
    prepared = requests.Request("GET", url, params=kwargs.get("params")).prepare()
    request_key = hashlib.sha256(prepared.url.encode("utf-8")).hexdigest()[:16]
    folder = RAW_DIR / "publisher_responses" / request_key
    if USE_PINNED_RAW:
        digest = PINNED_RESPONSES.get(request_key)
        if digest is None:
            raise FileNotFoundError("This publisher request has no pinned response")
        if len(digest) != 64 or any(ch not in "0123456789abcdef" for ch in digest):
            raise ValueError("Invalid pinned source checksum")
        body = (folder / f"{digest}.body").read_bytes()
        if hashlib.sha256(body).hexdigest() != digest:
            raise ValueError("Pinned publisher response checksum mismatch")
        metadata = json.loads((folder / f"{digest}.json").read_text(encoding="utf-8"))
        response = requests.Response()
        response.status_code = 200
        response.url = prepared.url
        response._content = body
    else:
        response = requests.get(url, timeout=90, **kwargs)
        response.raise_for_status()
        body = response.content
        if not body:
            raise ValueError("Publisher returned an empty response")
        digest = hashlib.sha256(body).hexdigest()
        folder.mkdir(parents=True, exist_ok=True)
        destination = folder / f"{digest}.body"
        if not destination.exists():
            destination.write_bytes(body)
        metadata = {"url": response.url, "sha256": digest, "bytes": len(body),
                    "accessed_at": datetime.now(UTC).isoformat()}
        (folder / f"{digest}.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    FETCH_LOG.append(metadata)
    return response

def parse_egrid(payload: bytes) -> pd.DataFrame:

    raw = pd.read_excel(BytesIO(payload), sheet_name="PLNT23", skiprows=1)
    generators = pd.read_excel(BytesIO(payload), sheet_name="GEN23", skiprows=1)
    raw = raw.merge(
        generator_technology(generators), on="ORISPL", how="left", validate="one_to_one"
    )
    candidates = {
        "plant_id": ["ORISPL", "Plant file sequence number"],
        "plant_name": ["PNAME", "Plant name"],
        "state": ["PSTATABB", "Plant state abbreviation"],
        "fuel": ["PLFUELCT"],
        "technology": ["technology"],
        "generation_mwh": ["PLNGENAN", "Plant annual net generation (MWh)"],
        "co2_tons": ["PLCO2AN", "Plant annual CO2 emissions (tons)"],
        "nox_tons": ["PLNOXAN", "Plant annual NOx emissions (tons)"],
        "so2_tons": ["PLSO2AN", "Plant annual SO2 emissions (tons)"],
    }
    out = {}
    for target, names in candidates.items():
        source = next((name for name in names if name in raw), None)
        if source:
            out[target] = raw[source]
    return pd.DataFrame(out)

def _egrid() -> pd.DataFrame:
    return parse_egrid(
        _get("https://www.epa.gov/system/files/documents/2025-06/egrid2023_data_rev2.xlsx").content
    )


Project: projects/power_plant_emissions
Publisher: https://www.epa.gov/system/files/documents/2025-06/egrid2023_data_rev2.xlsx
Pinned response folder: projects/power_plant_emissions/data/raw


## 2 Import

Run the import cell. It verifies the pinned response hash (or retrieves a new publisher response), parses it, and displays row count, columns, and a preview.

**Check before continuing:** no parse or source-integrity error; the imported table is nonempty, has the expected fields and grain, and the displayed source date matches the saved response metadata.


In [2]:
raw = _egrid()
if raw.empty:
    raise ValueError("Publisher returned no source records")
SNAPSHOT_DATE = SNAPSHOT_DATE_OVERRIDE or (
    str(FETCH_LOG[0].get("accessed_at") or "")[:10] if FETCH_LOG else date.today().isoformat()
)
raw.to_csv(RAW_DIR / "source_snapshot.csv", index=False)
print("Snapshot date:", SNAPSHOT_DATE)
print("Publisher responses:", len(FETCH_LOG))
print("Imported rows:", len(raw), "columns:", len(raw.columns))
display(raw.head(3))


Snapshot date: 2026-09-26
Publisher responses: 1
Imported rows: 12612 columns: 9


,plant_id,plant_name,state,fuel,technology,generation_mwh,co2_tons,nox_tons,so2_tons
0,57053,Alakanuk,AK,OIL,NaN,0.000,NaN,NaN,NaN
1,58982,Allison Creek Hydro,AK,HYDRO,HY,20961.000,NaN,NaN,NaN
2,60243,Ambler,AK,OIL,IC,1238.999,1125.234,22.889,1.998


## 3 Transform and clean

Run the original Power plant emissions transformation, embedded below. Inspect the fields, grain, and missing values before continuing.

**Check before continuing: inspect the cleaned preview, required fields, record grain, types, and missing values. Stop if a key or type rule fails.**


In [3]:
from dataclasses import asdict, dataclass

import pandas as pd


@dataclass(frozen=True)
class QualityReport:
    rows: int
    columns: int
    duplicate_rows: int
    missing_cells: int
    missing_rate: float
    passed: bool
    notes: list[str]


class DataQualityError(ValueError):
    """Raised when a dataset fails the required release checks."""


def profile(df: pd.DataFrame, key: str | None = None) -> dict:
    notes: list[str] = []
    duplicate_rows = int(df.duplicated().sum())
    missing_cells = int(df.isna().sum().sum())
    total_cells = max(df.shape[0] * df.shape[1], 1)
    if df.empty:
        notes.append("Dataset is empty")
    if key:
        if key not in df:
            notes.append(f"Required key '{key}' is missing")
        else:
            if df[key].isna().any():
                notes.append(f"Key '{key}' contains missing values")
            if df[key].duplicated().any():
                notes.append(f"Key '{key}' is not unique")
    if duplicate_rows:
        notes.append(f"Found {duplicate_rows} duplicate rows")
    report = QualityReport(
        rows=len(df),
        columns=len(df.columns),
        duplicate_rows=duplicate_rows,
        missing_cells=missing_cells,
        missing_rate=round(missing_cells / total_cells, 6),
        passed=not notes,
        notes=notes,
    )
    return asdict(report)


def require_passed(report: dict, project: str) -> None:
    if not report["passed"]:
        details = "; ".join(report["notes"]) or "unspecified quality failure"
        raise DataQualityError(f"Quality checks failed for {project}: {details}")


def require_columns(df: pd.DataFrame, columns: list[str]) -> None:
    missing = sorted(set(columns) - set(df.columns))
    if missing:
        raise ValueError(f"Required columns missing: {', '.join(missing)}")

def _snake(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = (
        out.columns.str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return out

def emissions(df: pd.DataFrame) -> pd.DataFrame:
    x = _snake(df)
    require_columns(x, ["plant_id", "plant_name", "state", "fuel", "generation_mwh", "co2_tons"])
    if x.plant_id.isna().any() or x.plant_id.duplicated().any():
        raise ValueError("Plant identifiers must be observed and unique")
    x["technology"] = x.get("technology", pd.Series("Unknown", index=x.index)).fillna("Unknown")
    for col in ["generation_mwh", "co2_tons", "nox_tons", "so2_tons"]:
        x[col] = pd.to_numeric(x.get(col), errors="coerce")
    x["co2_lb_mwh"] = (
        (2000 * x["co2_tons"] / x["generation_mwh"]).replace([np.inf, -np.inf], np.nan).round(2)
    )
    x["clean_generation_flag"] = (
        x["fuel"].astype(str).str.lower().isin(["wind", "solar", "hydro", "nuclear"]).astype(int)
    )
    x["co2_lb_mwh"] = x.co2_lb_mwh.where(x.generation_mwh.gt(0) & x.co2_tons.ge(0))
    scenarios = peer_scenarios(x.rename(columns={"generation_mwh": "net_generation_mwh"}))
    median = scenarios.loc[
        scenarios.benchmark_quantile.eq(0.5),
        ["plant_id", "benchmark", "peer_count", "scenario_co2_tons"],
    ]
    x = x.merge(median, on="plant_id", how="left", validate="one_to_one")
    x["abatement_opportunity_tons"] = x.pop("scenario_co2_tons")
    x["peer_benchmark_tons_mwh"] = x.pop("benchmark")
    # State generation share for context; this is not national or capacity share.
    x["generation_share_pct"] = (
        100 * x["generation_mwh"] / x.groupby("state")["generation_mwh"].transform("sum")
    ).round(2)
    # Review percentile, not a feasibility score; unavailable peers remain null.
    x["transition_priority_score"] = (100 * x.abatement_opportunity_tons.rank(pct=True)).round(1)
    return x[
        [
            "plant_id",
            "plant_name",
            "state",
            "fuel",
            "generation_mwh",
            "co2_tons",
            "nox_tons",
            "so2_tons",
            "co2_lb_mwh",
            "clean_generation_flag",
            "abatement_opportunity_tons",
            "generation_share_pct",
            "transition_priority_score",
            "technology",
            "peer_benchmark_tons_mwh",
            "peer_count",
        ]
    ]

clean = emissions(raw)
print("Clean rows:", len(clean), "columns:", len(clean.columns))
display(clean.head(3))


Clean rows: 12612 columns: 16


,plant_id,plant_name,state,fuel,generation_mwh,co2_tons,nox_tons,so2_tons,co2_lb_mwh,clean_generation_flag,abatement_opportunity_tons,generation_share_pct,transition_priority_score,technology,peer_benchmark_tons_mwh,peer_count
0,57053,Alakanuk,AK,OIL,0.000,NaN,NaN,NaN,NaN,0,NaN,0.00,NaN,Unknown,NaN,NaN
1,58982,Allison Creek Hydro,AK,HYDRO,20961.000,NaN,NaN,NaN,NaN,1,NaN,0.31,NaN,HY,0.010135,13.0
2,60243,Ambler,AK,OIL,1238.999,1125.234,22.889,1.998,1816.36,0,65.33692,0.02,66.3,IC,0.855446,494.0


## 4 Calculate

Run the original analytical formulas, embedded below. Inspect the peer_scenarios output before continuing.

**Check before continuing: confirm the named result table is nonempty and its labels and units match the project description. Do not treat a fresh snapshot as the historical baseline.**


In [4]:
"""Decision-specific analytical routines used by notebooks and release checks."""


import re
from dataclasses import dataclass

import numpy as np
import pandas as pd

SEED = 20260904


@dataclass(frozen=True)
class AnalysisSpec:
    group: str
    metric: str
    score: str
    entity: str
    date: str | None
    higher_is_risk: bool = True


SPECS = {
    "01_consumer_complaints": AnalysisSpec(
        "product", "timely_flag", "response_risk_score", "complaint_id", "date_received", False
    ),
    "03_sec_financial_health": AnalysisSpec(
        "bank_name", "noncurrent_loan_pct", "composite_stress_score", "cert", "report_date"
    ),
    "04_hospital_readmissions": AnalysisSpec(
        "measure_name", "shrunk_readmission_ratio", "priority_score", "facility_id", None
    ),
    "05_drug_safety": AnalysisSpec(
        "therapeutic_category",
        "shortage_age_days",
        "escalation_score",
        "package_ndc",
        "update_date",
    ),
    "07_electricity_affordability": AnalysisSpec(
        "state", "sales_yoy_pct", "load_pressure_index", "state", "period"
    ),
    "08_power_plant_emissions": AnalysisSpec(
        "fuel", "co2_lb_mwh", "transition_priority_score", "plant_id", None
    ),
    "10_semiconductor_supply_chain": AnalysisSpec(
        "metric", "yoy_pct", "pressure_zscore", "series_id", "period"
    ),
}

def data_audit(frame: pd.DataFrame, entity: str) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "check": [
                "rows",
                "columns",
                "duplicate_entity_rows",
                "missing_cells",
                "numeric_columns",
            ],
            "value": [
                len(frame),
                frame.shape[1],
                int(frame.duplicated(entity).sum()) if entity in frame else 0,
                int(frame.isna().sum().sum()),
                len(frame.select_dtypes(include="number").columns),
            ],
        }
    )

def segment_scorecard(frame: pd.DataFrame, spec: AnalysisSpec) -> pd.DataFrame:
    grouped = (
        frame.groupby(spec.group, dropna=False)
        .agg(
            records=(spec.entity, "size"),
            metric_mean=(spec.metric, "mean"),
            metric_median=(spec.metric, "median"),
            risk_score=(spec.score, "mean"),
        )
        .reset_index()
    )
    grouped["metric_gap_vs_total"] = grouped["metric_mean"] - frame[spec.metric].mean()
    grouped["risk_rank"] = (
        grouped["risk_score"].rank(method="dense", ascending=False).astype("Int64")
    )
    return grouped.sort_values(["risk_rank", "records"], ascending=[True, False])

def bootstrap_top_gap(frame: pd.DataFrame, spec: AnalysisSpec, draws: int = 1500) -> pd.DataFrame:
    rng = np.random.default_rng(SEED)
    scorecard = segment_scorecard(frame, spec)
    focus = scorecard.iloc[0][spec.group]
    a = frame.loc[frame[spec.group] == focus, spec.metric].dropna().to_numpy(float)
    b = frame.loc[frame[spec.group] != focus, spec.metric].dropna().to_numpy(float)
    if min(len(a), len(b)) < 2:
        return pd.DataFrame(
            {"focus_group": [focus], "gap": [np.nan], "ci_low": [np.nan], "ci_high": [np.nan]}
        )
    gaps = np.empty(draws)
    for i in range(draws):
        gaps[i] = (
            rng.choice(a, len(a), replace=True).mean() - rng.choice(b, len(b), replace=True).mean()
        )
    return pd.DataFrame(
        {
            "focus_group": [focus],
            "gap": [a.mean() - b.mean()],
            "ci_low": [np.quantile(gaps, 0.025)],
            "ci_high": [np.quantile(gaps, 0.975)],
            "bootstrap_draws": [draws],
        }
    )

def feature_diagnostics(frame: pd.DataFrame, spec: AnalysisSpec) -> pd.DataFrame:
    numeric = frame.select_dtypes(include="number").drop(columns=[spec.score], errors="ignore")
    numeric = numeric.loc[:, numeric.nunique(dropna=True) > 1]
    if spec.score not in frame or numeric.empty:
        return pd.DataFrame(columns=["feature", "spearman_with_risk", "absolute_association"])
    corr = numeric.corrwith(frame[spec.score], method="spearman").dropna()
    return (
        corr.rename("spearman_with_risk")
        .rename_axis("feature")
        .reset_index()
        .assign(absolute_association=lambda d: d["spearman_with_risk"].abs())
        .sort_values("absolute_association", ascending=False)
        .head(12)
    )

def temporal_diagnostics(frame: pd.DataFrame, spec: AnalysisSpec) -> pd.DataFrame:
    if not spec.date or spec.date not in frame:
        return pd.DataFrame(
            columns=["period", "records", "metric_mean", "risk_score", "rolling_risk"]
        )
    x = frame.dropna(subset=[spec.date]).copy()
    x["period"] = pd.to_datetime(x[spec.date]).dt.to_period("M").dt.to_timestamp()
    out = (
        x.groupby("period")
        .agg(
            records=(spec.entity, "size"),
            metric_mean=(spec.metric, "mean"),
            risk_score=(spec.score, "mean"),
        )
        .reset_index()
        .sort_values("period")
    )
    out["rolling_risk"] = out["risk_score"].rolling(3, min_periods=1).mean()
    out["risk_change"] = out["rolling_risk"].diff()
    return out

def sensitivity(frame: pd.DataFrame, spec: AnalysisSpec) -> pd.DataFrame:
    rows = []
    for quantile in [0.70, 0.75, 0.80, 0.85, 0.90]:
        threshold = frame[spec.score].quantile(quantile)
        flagged = frame[frame[spec.score] >= threshold]
        rows.append(
            {
                "risk_quantile": quantile,
                "threshold": threshold,
                "flagged_records": len(flagged),
                "flagged_share": len(flagged) / len(frame),
                "flagged_metric_mean": flagged[spec.metric].mean(),
            }
        )
    return pd.DataFrame(rows)

def decision_queue(frame: pd.DataFrame, spec: AnalysisSpec, limit: int = 20) -> pd.DataFrame:
    columns = [spec.entity, spec.group, spec.metric, spec.score]
    if spec.date:
        columns.insert(1, spec.date)
    columns = list(dict.fromkeys(c for c in columns if c in frame))
    return frame.nlargest(limit, spec.score)[columns].reset_index(drop=True)

def emissions_pareto(frame: pd.DataFrame) -> pd.DataFrame:
    x = frame.sort_values(
        ["abatement_opportunity_tons", "plant_id"], ascending=[False, True]
    ).copy()
    total = x.abatement_opportunity_tons.sum()
    x["cumulative_abatement_share"] = x["abatement_opportunity_tons"].cumsum() / (
        total if total > 0 else np.nan
    )
    x["priority_set"] = np.where(
        x.abatement_opportunity_tons.gt(0)
        & ((x.abatement_opportunity_tons.cumsum() - x.abatement_opportunity_tons) < 0.8 * total),
        "Top 80% opportunity",
        "Remaining",
    )
    x.loc[x.abatement_opportunity_tons.isna(), "priority_set"] = "Not estimable"
    return x[
        [
            "plant_id",
            "plant_name",
            "state",
            "fuel",
            "generation_mwh",
            "abatement_opportunity_tons",
            "cumulative_abatement_share",
            "priority_set",
        ]
    ]

def run_analysis(slug: str, frame: pd.DataFrame) -> dict[str, pd.DataFrame]:
    spec = SPECS[slug]
    outputs = {
        "data_audit": data_audit(frame, spec.entity),
        "segment_scorecard": segment_scorecard(frame, spec),
        "bootstrap_gap": bootstrap_top_gap(frame, spec),
        "feature_diagnostics": feature_diagnostics(frame, spec),
        "temporal_diagnostics": temporal_diagnostics(frame, spec),
        "threshold_sensitivity": sensitivity(frame, spec),
        "decision_queue": decision_queue(frame, spec),
    }
    if slug == "01_consumer_complaints":
        outputs["response_queue_intervals"] = monthly_queue(frame)
    elif slug == "03_sec_financial_health":

        outputs["deposit_outflow_scenario"] = bank_scenario(frame)
        outputs["threshold_sensitivity"] = threshold_sensitivity(frame)
    elif slug == "04_hospital_readmissions":
        outputs["observed_rate_uncertainty"] = observed_rate_review(frame)
        outputs["shrinkage_sensitivity"] = shrinkage_sensitivity(frame)
    elif slug == "05_drug_safety":
        outputs["concentration_analysis"] = shortage_concentration(frame)
        outputs["record_hhi_sensitivity"] = concentration_sensitivity(frame)
    elif slug == "07_electricity_affordability":
        outputs["growth_scenarios"] = energy_projection(frame)
        outputs["monthly_pressure_signals"] = monthly_signals(frame)
        outputs["persistence_sensitivity"] = persistence_sensitivity(frame)
    elif slug == "08_power_plant_emissions":
        outputs["abatement_pareto"] = emissions_pareto(frame)
        outputs["peer_scenarios"] = peer_scenarios(
            frame.rename(columns={"generation_mwh": "net_generation_mwh"}).drop(
                columns=["peer_count"], errors="ignore"
            )
        )
    elif slug == "10_semiconductor_supply_chain":
        outputs["divergence_analysis"] = supply_chain_divergence(frame)
        outputs.update(pulse_tables(frame))
    return outputs

analysis_outputs = run_analysis(SLUG, clean)
print("Calculated outputs:", {name: len(frame) for name, frame in analysis_outputs.items()})
display(analysis_outputs["peer_scenarios"].head(5))


Calculated outputs: {'data_audit': 5, 'segment_scorecard': 11, 'bootstrap_gap': 1, 'feature_diagnostics': 11, 'temporal_diagnostics': 0, 'threshold_sensitivity': 5, 'decision_queue': 20, 'abatement_pareto': 12612, 'peer_scenarios': 37836}


,plant_id,plant_name,state,fuel,net_generation_mwh,co2_tons,nox_tons,so2_tons,co2_lb_mwh,clean_generation_flag,...,generation_share_pct,transition_priority_score,technology,peer_benchmark_tons_mwh,co2_tons_per_mwh,peer_count,benchmark,scenario_co2_tons,benchmark_quantile,cumulative_opportunity_share
0,8042,Belews Creek,NC,GAS,8487941.0,6.118977e+06,7170.047,1624.177,1441.80,0,...,6.71,99.7,ST,0.594885,0.720902,173.0,0.329626,3.321131e+06,0.25,0.016593
1,628,Crystal River,FL,GAS,16086131.0,9.198521e+06,3970.340,3540.565,1143.66,0,...,6.23,100.0,Mixed,0.435318,0.571829,472.0,0.409163,2.616673e+06,0.25,0.029666
2,2721,Cliffside,NC,GAS,5244807.0,4.068164e+06,2022.290,1023.027,1551.31,0,...,4.15,99.6,ST,0.594885,0.775656,173.0,0.329626,2.339339e+06,0.25,0.041354
3,6146,Martin Lake,TX,COAL,11322156.0,1.399632e+07,9144.921,13302.559,2472.38,0,...,2.08,99.7,ST,1.133611,1.236188,210.0,1.046797,2.144324e+06,0.25,0.052068
4,6204,Laramie River,WY,COAL,9464926.0,1.200797e+07,6897.233,6630.412,2537.36,0,...,21.92,99.8,ST,1.133611,1.268681,210.0,1.046797,2.100117e+06,0.25,0.062560


## 5 Check

Run every assertion and inspect the quality report. These checks apply to the current response, not to unsupplied historical downloads.

**Check before continuing:** every assertion passes and `passed` is true. Investigate missing keys, duplicates, impossible values, and material row-count changes before release.


In [5]:
KEY_COLUMNS = ["plant_id"]
report = profile(clean)
assert report["passed"], report["notes"]
assert clean[KEY_COLUMNS].notna().all().all(), "Missing record key"
assert not clean.duplicated(KEY_COLUMNS).any(), "Duplicate record key"
assert len(raw) > 0 and len(clean) > 0
print("Current source rows:", len(raw), "clean rows:", len(clean))
print("Quality:", report)


Current source rows: 12612 clean rows: 12612
Quality: {'rows': 12612, 'columns': 16, 'duplicate_rows': 0, 'missing_cells': 60166, 'missing_rate': 0.298158, 'passed': True, 'notes': []}


## 6 Save

Run this cell only after the checks pass. It writes the dated extract, quality report, analytical tables, and source evidence under `data/manual_rebuild/`. If a fresh response has the same date as an existing rebuild, Save uses a source-digest suffix so the verified result is not overwritten; it refuses a conflicting partial folder.

**Check before continuing:** every named output exists in the dated folder; the source digest and reported row counts match the current response.


In [6]:
CURRENT_SOURCE_SHA = [download_report["sha256"]] if SLUG == "01_consumer_complaints" else [item["sha256"] for item in FETCH_LOG]
OUTPUT_DIR = PROJECT / "data" / "manual_rebuild" / SNAPSHOT_DATE
if OUTPUT_DIR.exists():
    prior_file = OUTPUT_DIR / "quality_report.json"
    prior_hashes = json.loads(prior_file.read_text(encoding="utf-8")).get("source_sha256") if prior_file.is_file() else None
    if isinstance(prior_hashes, str):
        prior_hashes = [prior_hashes]
    if prior_hashes != CURRENT_SOURCE_SHA:
        digest_suffix = hashlib.sha256(json.dumps(CURRENT_SOURCE_SHA).encode("utf-8")).hexdigest()[:12]
        OUTPUT_DIR = PROJECT / "data" / "manual_rebuild" / f"{SNAPSHOT_DATE}-{digest_suffix}"
        if OUTPUT_DIR.exists():
            raise FileExistsError(f"A different or partial rebuild already exists at {OUTPUT_DIR}; review it before saving")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
clean.to_csv(OUTPUT_DIR / "tableau_extract.csv", index=False, date_format="%Y-%m-%d %H:%M:%S")
report.update({"project": SLUG, "source": "EPA eGRID2023 revision 2",
               "sample_mode": False, "snapshot_date": SNAPSHOT_DATE,
               "source_sha256": [item["sha256"] for item in FETCH_LOG]})
(OUTPUT_DIR / "quality_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
analysis_dir = OUTPUT_DIR / "analysis"
analysis_dir.mkdir(exist_ok=True)
for name, frame in analysis_outputs.items():
    frame.to_csv(analysis_dir / f"{name}.csv", index=False, date_format="%Y-%m-%d")
(analysis_dir / "manifest.json").write_text(json.dumps({
    "project": SLUG, "sample_mode": False,
    "artifacts": {name: len(frame) for name, frame in analysis_outputs.items()}
}, indent=2), encoding="utf-8")
(OUTPUT_DIR / "source_manifest.json").write_text(json.dumps({
    "source_url": SOURCE_URL, "snapshot_date": SNAPSHOT_DATE,
    "responses": FETCH_LOG
}, indent=2), encoding="utf-8")
print("Saved manual rebuild to:", OUTPUT_DIR.relative_to(ROOT).as_posix())


Saved manual rebuild to: projects/power_plant_emissions/data/manual_rebuild/2026-09-26


### Manual MySQL handoff after Save

If you are demonstrating the CDC-style database step, open this project's `sql/README.md` now. In MySQL Workbench, run `schema.sql`, stage this run's checked `tableau_extract.csv`, run a temporary path-edited copy of `populate.sql`, then run `validation.sql` and `analysis.sql`. The SQL files create and check a mirror of the saved extract; they do not replace the notebook's calculations.

**Visible output:** Workbench shows the loaded row count, distinct source keys, missing-key checks, and a small inspection query. **Check before Show:** loaded rows equal this run's `quality_report.json` rows, distinct keys equal loaded rows, missing keys equal zero, and no unexplained load warnings remain. For a fresh response, use its own dated folder, not the pinned counts. If MySQL is unavailable, continue to Show with the notebook evidence and say explicitly that the SQL mirror was not tested.


## 7 Show

Display the rebuilt analytical table below. The packaged Tableau workbook in this project's tableau folder displays the pinned 2026-09-26 extract and is an additional saved view. Compare the saved table and dashboard before citing either; a fresh publisher response requires a deliberate workbook refresh and caption review.

**Check before continuing:** the displayed rows come from the current `analysis_outputs` and saved extract, with labels and units checked against this project's README.


In [7]:
display(analysis_outputs["peer_scenarios"].head(10))
print("Tableau input:", (OUTPUT_DIR / "tableau_extract.csv").relative_to(ROOT).as_posix())


,plant_id,plant_name,state,fuel,net_generation_mwh,co2_tons,nox_tons,so2_tons,co2_lb_mwh,clean_generation_flag,...,generation_share_pct,transition_priority_score,technology,peer_benchmark_tons_mwh,co2_tons_per_mwh,peer_count,benchmark,scenario_co2_tons,benchmark_quantile,cumulative_opportunity_share
0,8042,Belews Creek,NC,GAS,8.487941e+06,6.118977e+06,7170.047,1624.177,1441.80,0,...,6.71,99.7,ST,0.594885,0.720902,173.0,0.329626,3.321131e+06,0.25,0.016593
1,628,Crystal River,FL,GAS,1.608613e+07,9.198521e+06,3970.340,3540.565,1143.66,0,...,6.23,100.0,Mixed,0.435318,0.571829,472.0,0.409163,2.616673e+06,0.25,0.029666
2,2721,Cliffside,NC,GAS,5.244807e+06,4.068164e+06,2022.290,1023.027,1551.31,0,...,4.15,99.6,ST,0.594885,0.775656,173.0,0.329626,2.339339e+06,0.25,0.041354
3,6146,Martin Lake,TX,COAL,1.132216e+07,1.399632e+07,9144.921,13302.559,2472.38,0,...,2.08,99.7,ST,1.133611,1.236188,210.0,1.046797,2.144324e+06,0.25,0.052068
4,6204,Laramie River,WY,COAL,9.464926e+06,1.200797e+07,6897.233,6630.412,2537.36,0,...,21.92,99.8,ST,1.133611,1.268681,210.0,1.046797,2.100117e+06,0.25,0.062560
5,667,Northside,FL,GAS,3.191859e+06,3.068892e+06,2238.714,1505.690,1922.95,0,...,1.24,99.8,ST,0.594885,0.961475,173.0,0.329626,2.016773e+06,0.25,0.072636
6,2963,Northeastern,OK,GAS,3.489797e+06,3.439813e+06,2985.114,3188.085,1971.35,0,...,3.93,100.0,Mixed,0.435318,0.985677,472.0,0.409163,2.011918e+06,0.25,0.082688
7,3,Barry,AL,GAS,8.675796e+06,5.492460e+06,1325.515,280.451,1266.16,0,...,6.26,99.9,Mixed,0.435318,0.633078,472.0,0.409163,1.942645e+06,0.25,0.092394
8,55641,Riverside Energy Center,WI,GAS,3.543044e+06,3.150745e+06,155.526,15.873,1778.55,0,...,5.67,99.9,Mixed,0.435318,0.889276,472.0,0.409163,1.701063e+06,0.25,0.100893
9,6073,Daniel Electric Generating Plant,MS,GAS,9.488042e+06,5.572990e+06,3049.243,188.343,1174.74,0,...,13.01,99.9,Mixed,0.435318,0.587370,472.0,0.409163,1.690835e+06,0.25,0.109340


Tableau input: projects/power_plant_emissions/data/manual_rebuild/2026-09-26/tableau_extract.csv


## 8 Decide

The next cell records a source-dated decision, saved evidence, and an interpretation limit for the pinned response. Review and revise both statements if you acquire a fresh response; do not carry this conclusion forward automatically.

**Final check:** the current quality check passed, each evidence file exists, and the conclusion is no stronger than the analysis supports.


In [9]:
DECISION_TEXT = (
    "For the pinned 2026-09-26 EPA eGRID snapshot, prioritize manual "
    "evidence review of Crystal River and the 205-plant set labeled Top 80% "
    "opportunity. Crystal River leads the saved abatement Pareto at about "
    "2.20 million modeled tons among 3,233 plants with estimable scenarios. "
    "This is a screening queue, not an estimate of achievable reductions."
)
INTERPRETATION_LIMIT = (
    "Peer benchmarks do not establish retrofit feasibility or future dispatch. "
    "Missing generation or emissions leave many plants without an estimable "
    "scenario; fuel, technology, and reporting-year differences limit comparisons."
)
EVIDENCE_FILES = [
    "quality_report.json",
    "analysis/abatement_pareto.csv",
    "analysis/peer_scenarios.csv",
]
assert report["passed"], "Prior quality check must pass"
assert DECISION_TEXT.strip(), "Write a decision supported by the rebuilt evidence"
assert INTERPRETATION_LIMIT.strip(), "State an interpretation limit"
missing = [name for name in EVIDENCE_FILES if not (OUTPUT_DIR / name).is_file()]
assert not missing, f"Missing evidence files: {missing}"
print("Source snapshot:", SNAPSHOT_DATE)
print("Evidence:", EVIDENCE_FILES)
print("Decision:", DECISION_TEXT)
print("Limit:", INTERPRETATION_LIMIT)

Source snapshot: 2026-09-26
Evidence: ['quality_report.json', 'analysis/abatement_pareto.csv', 'analysis/peer_scenarios.csv']
Decision: For the pinned 2026-09-26 EPA eGRID snapshot, prioritize manual evidence review of Crystal River and the 205-plant set labeled Top 80% opportunity. Crystal River leads the saved abatement Pareto at about 2.20 million modeled tons among 3,233 plants with estimable scenarios. This is a screening queue, not an estimate of achievable reductions.
Limit: Peer benchmarks do not establish retrofit feasibility or future dispatch. Missing generation or emissions leave many plants without an estimable scenario; fuel, technology, and reporting-year differences limit comparisons.
